# k06 — Recording-level audit of the frozen k05 B-cell results
Post-hoc statistical audit only: no training, no re-scoring, no change to the frozen design, no recording excluded from the results.
Per B cell: per-recording positives/prevalence/PR-AUC (diagnostic; unstable at low positive counts), leave-one-recording-out recomputation of the cell metric (replaces the ill-defined per-recording "contribution" to a global ranking metric), the **exact** token-stratified recording bootstrap by complete enumeration (replaces 2,000 random draws — same estimator, no resolution floor), seed vs recording variability, and the B summary / manufacturer strata from the exact per-cell distributions.

In [ ]:
import os, glob
os.makedirs('/kaggle/working/code', exist_ok=True)
open('/kaggle/working/code/exp_audit.py','w').write('"""Stage 5c recording-level audit of the FROZEN k05 outputs. No training, no re-scoring, no exclusion of data.\nPurpose: test whether B-cell conclusions are robust to recording-level imbalance (2 recordings per attack token; one can dominate positives).\nUsage: python exp_audit.py --eval <k05 eval dir> --out /kaggle/working/audit"""\nimport os, sys, json, glob, argparse, itertools, math\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nCONTRASTS = {\'H1_GS_minus_DeepSets\': (\'GraphSAGE\', \'DeepSets\'), \'H1a_GS_minus_rewired\': (\'GraphSAGE\', \'GraphSAGE_rewired\'),\n             \'H2_GS_minus_GRU\': (\'GraphSAGE\', \'GRU\'), \'H3_GS_minus_LGBS\': (\'GraphSAGE\', \'LightGBM_S\'),\n             \'ladder_LGBS_minus_LGB\': (\'LightGBM_S\', \'LightGBM\')}\nMARGIN = 0.02; MC_SUMMARY = 200000; MC_SEED = 20260917\n\ndef ap_prep(s, y, fid, nfiles):\n    o = np.argsort(-s, kind=\'stable\'); ss = s[o]; ys = y[o].astype(np.float64); fs = fid[o]\n    ends = np.r_[np.flatnonzero(ss[1:] != ss[:-1]), len(ss) - 1]\n    CP = np.zeros((nfiles, len(ends))); CN = np.zeros((nfiles, len(ends)))\n    for f in range(nfiles):\n        m = fs == f\n        CP[f] = np.cumsum(m * ys)[ends]; CN[f] = np.cumsum(m * (1 - ys))[ends]\n    return CP, CN\n\ndef ap_w(CP, CN, W, chunk=64):\n    out = np.empty(len(W))\n    for c in range(0, len(W), chunk):\n        w = W[c:c + chunk]\n        tp = w @ CP; fp = w @ CN; P = tp[:, -1:]; den = tp + fp\n        prec = np.where(den > 0, tp / np.where(den > 0, den, 1), 0.0)\n        rec = np.where(P > 0, tp / np.where(P > 0, P, 1), 0.0)\n        ap = (np.diff(rec, axis=1, prepend=0.0) * prec).sum(1)\n        out[c:c + chunk] = np.where(P[:, 0] > 0, ap, np.nan)\n    return out\n\ndef exact_token_resamples(tokens):\n    """Every distinct token-stratified bootstrap outcome with its probability. For a token with k recordings, the\n    multiset of k draws with replacement; probability = multinomial weight / k**k."""\n    tokens = np.asarray(tokens); per = []\n    for t in sorted(set(tokens.tolist())):\n        idx = np.flatnonzero(tokens == t); k = len(idx); opts = {}\n        for draw in itertools.product(range(k), repeat=k):\n            cnt = tuple(sorted(np.bincount(draw, minlength=k).tolist(), reverse=False))\n            key = tuple(np.bincount(draw, minlength=k).tolist())\n            opts[key] = opts.get(key, 0) + 1\n        per.append([(idx, np.array(key, float), c / float(k ** k)) for key, c in opts.items()])\n    W = []; P = []\n    for combo in itertools.product(*per):\n        w = np.zeros(len(tokens)); p = 1.0\n        for idx, cnt, pr in combo:\n            w[idx] = cnt; p *= pr\n        W.append(w); P.append(p)\n    return np.array(W), np.array(P)\n\ndef wq(vals, probs, q):\n    o = np.argsort(vals); v = vals[o]; c = np.cumsum(probs[o]); c /= c[-1]\n    return float(v[np.searchsorted(c, q, side=\'left\').clip(0, len(v) - 1)])\n\ndef summarise(vals, probs):\n    m = np.isfinite(vals); vals, probs = vals[m], probs[m] / probs[m].sum()\n    lo95, hi95, lo90, hi90 = wq(vals, probs, .025), wq(vals, probs, .975), wq(vals, probs, .05), wq(vals, probs, .95)\n    if lo95 > 0: d = \'superior\'\n    elif hi95 < 0: d = \'inferior\'\n    elif lo90 >= -MARGIN and hi90 <= MARGIN: d = \'practically_equivalent\'\n    else: d = \'inconclusive\'\n    p = min(1.0, 2 * min(probs[vals <= 0].sum(), probs[vals >= 0].sum()))\n    return {\'mean_over_resamples\': float((vals * probs).sum()), \'ci95\': [lo95, hi95], \'ci90\': [lo90, hi90],\n            \'decision\': d, \'p_exact_two_sided\': float(p), \'n_distinct_resamples\': int(len(vals))}\n\ndef main(EVAL, OUT):\n    os.makedirs(OUT, exist_ok=True); R = {\'cells\': {}, \'summary\': {}, \'method\': {\n        \'per_recording_ap\': \'AP computed on that recording alone (diagnostic only; unstable at low positive counts)\',\n        \'leave_one_out\': \'cell AP recomputed with that recording removed (replaces the ill-defined per-recording contribution to a global ranking metric)\',\n        \'exact_bootstrap\': \'complete enumeration of the token-stratified recording bootstrap with exact probabilities (replaces 2,000 random draws; same estimator)\',\n        \'no_exclusions\': \'no recording is dropped from the frozen results; leave-one-out is diagnostic\'}}\n    per_cell_dist = {}\n    for path in sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'test_02_scores.npz\'))):\n        st = path.split(os.sep)[-2]; key = f\'{st}/test_02\'\n        Z = np.load(path); meta = json.load(open(path.replace(\'_scores.npz\', \'_meta.json\')))\n        y = Z[\'y\'].astype(np.int8); fid = Z[\'file_id\'].astype(np.int64); files = meta[\'files\']; nf = len(files)\n        tokens = [f[\'token\'] for f in files]\n        S = {m: Z[f\'score_{m}\'].astype(np.float64) for m in MODELS}\n        cell = {\'files\': [], \'aggregate\': {}, \'leave_one_out\': {}, \'exact_bootstrap\': {}, \'seed_vs_recording_variability\': {}}\n        tot_pos = int(y.sum())\n        for i, f in enumerate(files):\n            ix = fid == i; rec = {\'file\': f[\'file\'], \'token\': f[\'token\'], \'windows\': int(ix.sum()), \'pos\': int(y[ix].sum()),\n                                  \'share_of_cell_positives\': round(float(y[ix].sum()) / max(tot_pos, 1), 4),\n                                  \'prevalence\': round(float(y[ix].mean()), 5), \'hours\': round(f[\'hours\'], 4), \'ap_mean\': {}, \'ap_sd\': {}}\n            for m in MODELS:\n                a = [average_precision_score(y[ix], s[ix]) if y[ix].sum() > 0 else float(\'nan\') for s in S[m]]\n                rec[\'ap_mean\'][m] = round(float(np.mean(a)), 4); rec[\'ap_sd\'][m] = round(float(np.std(a, ddof=1)) if len(a) > 1 else 0.0, 4)\n            rec[\'contrasts\'] = {h: round(rec[\'ap_mean\'][a] - rec[\'ap_mean\'][b], 4) for h, (a, b) in CONTRASTS.items()}\n            cell[\'files\'].append(rec)\n        agg = {m: float(np.mean([average_precision_score(y, s) for s in S[m]])) for m in MODELS}\n        cell[\'aggregate\'] = {\'ap_mean\': {m: round(agg[m], 4) for m in MODELS},\n                             \'contrasts\': {h: round(agg[a] - agg[b], 4) for h, (a, b) in CONTRASTS.items()},\n                             \'windows\': int(len(y)), \'pos\': tot_pos, \'prevalence\': round(float(y.mean()), 5)}\n        for i, f in enumerate(files):   # leave-one-recording-out\n            keep = fid != i\n            if y[keep].sum() == 0: continue\n            a2 = {m: float(np.mean([average_precision_score(y[keep], s[keep]) for s in S[m]])) for m in MODELS}\n            cell[\'leave_one_out\'][f[\'file\']] = {\'ap_mean\': {m: round(a2[m], 4) for m in MODELS},\n                                                \'contrasts\': {h: round(a2[a] - a2[b], 4) for h, (a, b) in CONTRASTS.items()},\n                                                \'delta_vs_full\': {h: round((a2[a] - a2[b]) - (agg[a] - agg[b]), 4) for h, (a, b) in CONTRASTS.items()}}\n        W, P = exact_token_resamples(tokens)\n        B = {}\n        for m in MODELS:\n            acc = np.zeros(len(W))\n            for s in S[m]:\n                CP, CN = ap_prep(s, y, fid, nf)\n                chk = ap_w(CP, CN, np.ones((1, nf)))[0]\n                assert abs(chk - average_precision_score(y, s)) < 1e-9\n                acc += ap_w(CP, CN, W)\n            B[m] = acc / len(S[m])\n        per_cell_dist[key] = {\'W_prob\': P, \'ap\': B}\n        for h, (a, b) in CONTRASTS.items():\n            cell[\'exact_bootstrap\'][h] = {\'point_full_sample\': round(agg[a] - agg[b], 4), **summarise(B[a] - B[b], P)}\n        for m in MODELS:\n            sd_seed = float(np.std([average_precision_score(y, s) for s in S[m]], ddof=1)) if len(S[m]) > 1 else 0.0\n            sd_rec = float(np.sqrt(((B[m] - (B[m] * P).sum()) ** 2 * P).sum()))\n            cell[\'seed_vs_recording_variability\'][m] = {\'sd_across_seeds\': round(sd_seed, 4), \'sd_across_recording_resamples\': round(sd_rec, 4)}\n        R[\'cells\'][key] = cell\n        print(key, \'exact resamples\', len(W), \'aggregate\', cell[\'aggregate\'][\'contrasts\'], flush=True)\n    keys = sorted(per_cell_dist)\n    rng = np.random.default_rng(MC_SEED)\n    draws = {k: rng.choice(len(per_cell_dist[k][\'W_prob\']), size=MC_SUMMARY, p=per_cell_dist[k][\'W_prob\'] / per_cell_dist[k][\'W_prob\'].sum()) for k in keys}\n    strata = {\'B_summary\': keys, \'B_same_manufacturer\': [k for k in keys if k.startswith(\'set_01\')], \'B_cross_manufacturer\': [k for k in keys if not k.startswith(\'set_01\')]}\n    unif = np.full(MC_SUMMARY, 1.0 / MC_SUMMARY)\n    for h, (a, b) in CONTRASTS.items():\n        for sname, ks in strata.items():\n            d = np.mean([per_cell_dist[k][\'ap\'][a][draws[k]] - per_cell_dist[k][\'ap\'][b][draws[k]] for k in ks], 0)\n            pt = float(np.mean([R[\'cells\'][k][\'aggregate\'][\'contrasts\'][h] for k in ks]))\n            R[\'summary\'].setdefault(h, {})[sname] = {\'point_full_sample\': round(pt, 4), \'cells\': ks,\n                                                     **summarise(d, unif), \'note\': f\'{MC_SUMMARY} draws from the EXACT per-cell resample distributions\'}\n    json.dump(R, open(os.path.join(OUT, \'audit.json\'), \'w\'), indent=1)\n    # readable tables\n    L = []\n    for k, c in R[\'cells\'].items():\n        L.append(f"\\n== {k}  windows {c[\'aggregate\'][\'windows\']}  pos {c[\'aggregate\'][\'pos\']}  prevalence {c[\'aggregate\'][\'prevalence\']}")\n        L.append(\'recording\\ttoken\\twin\\tpos\\tposshare\\t\' + \'\\t\'.join(MODELS) + \'\\tGS-DS\\tGS-rew\')\n        for f in c[\'files\']:\n            L.append(f"{f[\'file\']}\\t{f[\'token\']}\\t{f[\'windows\']}\\t{f[\'pos\']}\\t{f[\'share_of_cell_positives\']}\\t" + \'\\t\'.join(f"{f[\'ap_mean\'][m]:.3f}" for m in MODELS)\n                     + f"\\t{f[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{f[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'AGGREGATE\\t\\t\\t\\t\\t\' + \'\\t\'.join(f"{c[\'aggregate\'][\'ap_mean\'][m]:.3f}" for m in MODELS)\n                 + f"\\t{c[\'aggregate\'][\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{c[\'aggregate\'][\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'leave-one-out (cell AP without that recording) — GS-DS / GS-rewired, and change vs full cell:\')\n        for fn, v in c[\'leave_one_out\'].items():\n            L.append(f"  drop {fn}\\tGS-DS {v[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f} ({v[\'delta_vs_full\'][\'H1_GS_minus_DeepSets\']:+.3f})"\n                     f"\\tGS-rew {v[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f} ({v[\'delta_vs_full\'][\'H1a_GS_minus_rewired\']:+.3f})"\n                     f"\\tGS-LGB+S {v[\'contrasts\'][\'H3_GS_minus_LGBS\']:+.3f} ({v[\'delta_vs_full\'][\'H3_GS_minus_LGBS\']:+.3f})")\n        L.append(\'exact token-stratified bootstrap:\')\n        for h, v in c[\'exact_bootstrap\'].items():\n            L.append(f"  {h}\\tpoint {v[\'point_full_sample\']:+.4f}\\tmean {v[\'mean_over_resamples\']:+.4f}\\tci95 [{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\tci90 [{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\tp {v[\'p_exact_two_sided\']:.4f}\\tN {v[\'n_distinct_resamples\']}")\n        L.append(\'variability (SD across seeds | SD across recording resamples):\')\n        L.append(\'  \' + \'  \'.join(f"{m} {v[\'sd_across_seeds\']:.3f}|{v[\'sd_across_recording_resamples\']:.3f}" for m, v in c[\'seed_vs_recording_variability\'].items()))\n    L.append(\'\\n== B summary / strata (exact per-cell distributions)\')\n    L.append(\'contrast\\tstratum\\tpoint\\tmean\\tci95\\tci90\\tdecision\\tp\')\n    for h, dd in R[\'summary\'].items():\n        for sname, v in dd.items():\n            L.append(f"{h}\\t{sname}\\t{v[\'point_full_sample\']:+.4f}\\t{v[\'mean_over_resamples\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t[{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\t{v[\'p_exact_two_sided\']:.4f}")\n    open(os.path.join(OUT, \'audit.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--eval\', required=True); ap.add_argument(\'--out\', default=\'/kaggle/working/audit\'); a = ap.parse_args()\n    main(a.eval, a.out)\n')
EV = sorted(glob.glob('/kaggle/input/**/eval/set_0*/test_02_scores.npz', recursive=True))
print(EV); assert len(EV) == 4
EVAL = os.path.dirname(os.path.dirname(EV[0])); print('eval dir', EVAL)


In [ ]:
import subprocess, sys, time
t0 = time.time()
r = subprocess.run([sys.executable, '/kaggle/working/code/exp_audit.py', '--eval', EVAL, '--out', '/kaggle/working/audit'])
print('exit', r.returncode, 'min', round((time.time() - t0) / 60, 2))


In [ ]:
print(open('/kaggle/working/audit/audit.tsv').read())